# Fix Bridge Table - Feedwater Flow Tag Mappings

**Issue:** Tag `RV3_FWFU3WF25_AG` (West pump) incorrectly associated with East pump predictions

**Solution:** Add explicit mappings for feedwater flow tags to ensure proper asset association

In [ ]:
# Step 1: Check current state of feedwater flow tags in PI metadata
print("=== FEEDWATER FLOW TAGS IN PI METADATA ===")
feedwater_tags = spark.sql("""
    SELECT Name, Descriptor, EngineeringUnits
    FROM dbo.pi_tags_metadata
    WHERE Name LIKE '%FWFU3WF%'
    ORDER BY Name
""")
feedwater_tags.show(50, False)

In [ ]:
# Step 2: Check if these tags exist in bridge table
print("\n=== CURRENT BRIDGE TABLE MAPPINGS FOR FEEDWATER TAGS ===")
current_mappings = spark.sql("""
    SELECT Tag, asset_id, tag_role, tag_description
    FROM gold.bridge_pi_tag_to_asset
    WHERE Tag LIKE '%FWFU3WF%'
    ORDER BY Tag
""")
current_mappings.show(50, False)
print(f"Count: {current_mappings.count()}")

In [ ]:
# Step 3: Check what assets exist for RV3 pumps
print("\n=== EXISTING RV3 PUMP ASSETS ===")
pump_assets = spark.sql("""
    SELECT DISTINCT asset_id
    FROM gold.bridge_pi_tag_to_asset
    WHERE asset_id LIKE '%RV3%' AND asset_id LIKE '%Pump%'
    ORDER BY asset_id
""")
pump_assets.show(50, False)

In [ ]:
# Step 4: Create proper mappings for feedwater flow tags
# Map tags ending in 24 (3E) to East pump, tags ending in 25 (3W) to West pump

from pyspark.sql import Row
from datetime import datetime

# Define new mappings based on tag naming convention
new_mappings = [
    # East Pump tags (3E / 24)
    Row(
        Tag="RV3:FWFU3WF24.AG",
        asset_id="RV3_U3_Boiler_Feed_Pump_East",
        tag_role="operational",
        downtime_relevance="HIGH",
        eng_units="KLB/Hr",
        tag_description="FEEDWATER FLOW THROUGH BFP 3E",
        notes="Explicitly mapped - East pump feedwater flow"
    ),
    Row(
        Tag="RV3:FWFU3WF20X.AG",
        asset_id="RV3_U3_Boiler_Feed_Pump_East",
        tag_role="operational",
        downtime_relevance="HIGH",
        eng_units="KLB/Hr",
        tag_description="FEEDWATER FLOW COMPENSATED",
        notes="Explicitly mapped - Compensated feedwater flow for East pump"
    ),
]

# Note: We're NOT mapping the West pump tags (25/3W) yet since there's no West pump asset
# They will be excluded from East pump features automatically

new_mappings_df = spark.createDataFrame(new_mappings)
print("\n=== NEW MAPPINGS TO ADD ===")
new_mappings_df.show(50, False)

In [ ]:
# Step 5: Check for conflicts before inserting
print("\n=== CHECKING FOR CONFLICTS ===")
tags_to_add = [row.Tag for row in new_mappings]
conflicts = spark.sql(f"""
    SELECT Tag, asset_id, tag_description
    FROM gold.bridge_pi_tag_to_asset
    WHERE Tag IN ({','.join(["'" + t + "'" for t in tags_to_add])})
""")
conflict_count = conflicts.count()
if conflict_count > 0:
    print(f"⚠️ Found {conflict_count} conflicting mappings:")
    conflicts.show(50, False)
    print("\nThese will need to be updated instead of inserted.")
else:
    print("✅ No conflicts - safe to insert")

In [ ]:
# Step 6: Insert new mappings (or update if conflicts exist)
print("\n=== APPLYING MAPPINGS ===")

# Create temp view for merge operation
new_mappings_df.createOrReplaceTempView("new_feedwater_mappings")

# Use merge to handle both inserts and updates
spark.sql("""
    MERGE INTO gold.bridge_pi_tag_to_asset AS target
    USING new_feedwater_mappings AS source
    ON target.Tag = source.Tag
    WHEN MATCHED THEN UPDATE SET
        target.asset_id = source.asset_id,
        target.tag_role = source.tag_role,
        target.downtime_relevance = source.downtime_relevance,
        target.eng_units = source.eng_units,
        target.tag_description = source.tag_description,
        target.notes = source.notes
    WHEN NOT MATCHED THEN INSERT (
        Tag, asset_id, tag_role, downtime_relevance, eng_units, tag_description, notes
    ) VALUES (
        source.Tag, source.asset_id, source.tag_role, source.downtime_relevance,
        source.eng_units, source.tag_description, source.notes
    )
""")

print("✅ Mappings applied successfully")

In [ ]:
# Step 7: Verify the fix
print("\n=== VERIFICATION - UPDATED BRIDGE TABLE ===")
updated_mappings = spark.sql("""
    SELECT Tag, asset_id, tag_role, tag_description, notes
    FROM gold.bridge_pi_tag_to_asset
    WHERE Tag LIKE '%FWFU3WF%'
    ORDER BY Tag
""")
updated_mappings.show(50, False)

## Next Steps

1. ✅ Bridge table updated with correct feedwater flow tag mappings
2. **Re-run Phase 2 Feature Engineering** to rebuild training dataset with corrected mappings
3. **Retrain Cox model** (Phase 3) with corrected features
4. **Re-score predictions** (Phase 4) to get accurate risk assessments

**Expected Impact:**
- East pump will no longer use West pump feedwater flow tags
- Prediction accuracy should improve since features now match the target asset
- Risk rankings may change significantly for RV3_U3_Boiler_Feed_Pump_East

In [ ]:
# Optional: Create exclusion list for West pump tags
# These tags should NOT be used for East pump features until a West pump asset is created
print("\n=== WEST PUMP TAGS TO EXCLUDE FROM EAST PUMP ===")
west_pump_tags = spark.sql("""
    SELECT Name, Descriptor, EngineeringUnits
    FROM dbo.pi_tags_metadata
    WHERE (Descriptor LIKE '%3W%' OR Descriptor LIKE '%WEST%')
      AND Name LIKE '%RV3%'
      AND Name LIKE '%FWF%'
    ORDER BY Name
""")
west_pump_tags.show(50, False)

print("\nThese tags should be excluded from RV3_U3_Boiler_Feed_Pump_East features")
print("Consider creating a RV3_U3_Boiler_Feed_Pump_West asset if this equipment exists")